# Collisions between charged particles

`ChargedCollisions` pairs the markers of charged species within cells, scatters
each pair with the Coulomb collision model of Takizuka and Abe, and lets pairs
react. This tutorial checks both against closed-form results: the Spitzer rate
at which two temperatures relax, and the rate equation of mutual
neutralization.

In [ ]:
import matplotlib.pyplot as plt

# light-mode categorical palette, used in this fixed order
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "lines.linewidth": 2.0,
        "lines.markersize": 6,
        "legend.frameon": False,
        "figure.figsize": (7.0, 4.2),
        "figure.dpi": 110,
        "font.size": 11,
        "axes.prop_cycle": plt.cycler(color=COLORS),
    }
)

In [ ]:
import math

import numpy as np

import plasmacoll
from plasmacoll.charged import VACUUM_PERMITTIVITY
from plasmacoll.constants import ATOMIC_MASS, ELEMENTARY_CHARGE

m_a, m_b = 1.00728 * ATOMIC_MASS, 4.0026 * ATOMIC_MASS  # H+ and He+
density, log = 1e20, 10.0


def spitzer(T_a, T_b):
    # dT_a/dt = nu (T_b - T_a), temperatures in eV
    return (
        8
        * math.sqrt(2 * math.pi)
        * density
        * ELEMENTARY_CHARGE**4
        * log
        * math.sqrt(m_a * m_b)
        / (
            3
            * (4 * math.pi * VACUUM_PERMITTIVITY) ** 2
            * ELEMENTARY_CHARGE**1.5
            * (m_a * T_b + m_b * T_a) ** 1.5
        )
    )


def maxwellian(num, mass, temperature, rng):
    # exactly the requested temperature, so that the start is the same for every run
    v = rng.standard_normal((num, 3))
    v -= v.mean(axis=0)
    v *= math.sqrt(temperature * ELEMENTARY_CHARGE / mass) / v.std()
    return plasmacoll.ParticleArrays(np.zeros((num, 1)), v, np.ones(num))


def temperature(markers, mass):
    return mass * (markers.velocities**2).sum(axis=1).mean() / (3 * ELEMENTARY_CHARGE)

## Temperature relaxation

H⁺ at 2 eV and He⁺ at 1 eV, at equal densities, relax towards each other at the
Spitzer rate $\nu_\varepsilon$. Points are the simulation with all three species
pairs colliding, the line the Spitzer rate equation integrated in time.

The simulation relaxes more slowly than Spitzer's rate, and the gap closes only
slowly as the time step shrinks: after one relaxation time the temperature has
changed by about 0.82 of the Spitzer value at $\nu_\varepsilon\Delta t = 10^{-2}$
and 0.87 at $10^{-3}$. Binary-collision methods are known to converge slowly in
$\Delta t$; see the
[accuracy of Coulomb collisions](/plasmacoll/guides/charged-collisions/#accuracy-and-the-time-step)
for what has and has not been traced.

In [ ]:
num = 20_000
rate0 = spitzer(2.0, 1.0)
fig, ax = plt.subplots()
for color, factor in zip(COLORS, (1e-2, 1e-3), strict=False):
    rng = np.random.default_rng(1)
    species = {"a": maxwellian(num, m_a, 2.0, rng), "b": maxwellian(num, m_b, 1.0, rng)}
    collisions = plasmacoll.ChargedCollisions(
        {"a": m_a, "b": m_b},
        {"a": 1, "b": 1},
        coulomb_pairs="all",
        coulomb_logarithm=log,
        seed=2,
    )
    dt = factor / rate0
    steps = int(round(1.0 / factor))
    every = max(steps // 25, 1)
    times, values = [0.0], [2.0]
    for step in range(1, steps + 1):
        collisions.collide_species(species, dt, cell_volume=num / density)
        if step % every == 0:
            times.append(step * dt * rate0)
            values.append(temperature(species["a"], m_a))
    ax.plot(
        times,
        values,
        "o",
        ms=4,
        color=color,
        label=rf"$\nu_\varepsilon\Delta t = {factor:g}$",
    )

# the Spitzer rate equation (equal densities: the mean temperature is constant)
T_a, T_b, model_t, model = 2.0, 1.0, [], []
step = 1e-3 / rate0
for k in range(1001):
    model_t.append(k * step * rate0)
    model.append(T_a)
    change = spitzer(T_a, T_b) * (T_b - T_a) * step
    T_a, T_b = T_a + change, T_b - change
ax.plot(model_t, model, "--", color=INK, label="Spitzer")
ax.set(
    xlabel=r"$\nu_\varepsilon t$", ylabel=r"$T_{H^+}$ (eV)", title="H⁺ cooling on He⁺"
)
ax.legend(frameon=False)
fig.tight_layout()

## Mutual neutralization

$\mathrm{H^-} + \mathrm{H^+} \to 2\mathrm{H}$ with a constant rate coefficient
$K$ follows $\dot n_- = \dot n_+ = -K n_- n_+$. With $\Delta = n_{+0} - n_{-0}$
conserved, $n_-(t) = \Delta / ((n_{+0}/n_{-0})\,e^{K\Delta t} - 1)$. The two
species have different marker weights here: a reaction removes the smaller
weight from both markers, so the densities stay exact in expectation and the
charge exactly.

In [ ]:
K, volume = 4e-13, 1e-6
n_minus, n_plus = 1e17, 2e17
masses = {"H-": m_a, "H+": m_a, "H": m_a}
reaction = plasmacoll.ChargedReaction(
    "mutual_neutralization",
    ("H-", "H+"),
    rate_coefficient=K,
    products={"neutral_1": "H", "neutral_2": "H"},
)
charged = plasmacoll.ChargedCollisions(
    masses, {"H-": -1, "H+": 1}, reactions=[reaction], seed=3
)
rng = np.random.default_rng(4)
species = {
    "H-": maxwellian(20_000, m_a, 0.1, rng),
    "H+": maxwellian(30_000, m_a, 0.1, rng),
}
species["H-"].weights *= n_minus * volume / 20_000
species["H+"].weights *= n_plus * volume / 30_000
reactor = plasmacoll.ZeroDReactor(
    plasmacoll.MonteCarloCollisions(masses, [], {}, seed=5),
    species,
    volume=volume,
    charged=charged,
)
history = reactor.run(0.01 / (K * n_plus), 600, every=20)
delta = n_plus - n_minus
expected = delta / ((n_plus / n_minus) * np.exp(K * delta * history.time) - 1)

fig, ax = plt.subplots()
t = history.time * K * n_plus
ax.plot(t, history.density["H-"], "o", ms=4, color=COLORS[0], label="H⁻")
ax.plot(t, history.density["H+"], "o", ms=4, color=COLORS[1], label="H⁺")
ax.plot(t, history.density["H"], "o", ms=4, color=COLORS[2], label="H")
ax.plot(t, expected, "--", color=INK, label="rate equation")
ax.set(
    xlabel=r"$K n_{+0} t$", ylabel=r"density (m$^{-3}$)", title="Mutual neutralization"
)
ax.legend(frameon=False, loc="center right")
fig.tight_layout()
print(
    "rate coefficient from the counts:",
    history.rate_coefficient(reaction.name, "H-", "H+"),
)

The rate coefficient recovered from the reaction counts by
`ReactorHistory.rate_coefficient` matches the $K$ that went in, which is how the
same diagnostic measures the rates of a cross-section set.